### Analysis of voltage dynamics during passive DoC

#### Notebook goals
- Validate extracted voltage data for each session, DMD, and ROI -- ensure quality alignment to image presentations
- Recapitulate analyses of image presentation dynamics (response variance, sequence dynamics) for voltage as in glutamate
- Summarize voltage dynamics per session, create a comprehensive idea of voltage dynamics correlations between ROIs on the same DMD and between DMDs
#### Specific analyses
- Image selectivity analyses x session x depth (variance, FVE, % dFF)
- Image sequence analysis
- Image change and omission analyses -- differences between depths, sessions, FVE by change relative to mean image response
- Voltage oscillation analysis
#### Order of operations
1) Gather relevant sessions -- sort, organize, and open
2) Plot example image PSTH for each ROI, DMD, session
3) Perform 


In [13]:
import sys
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [3]:
# ---------------------------------------------------------------------
# Local repository / data paths
# ---------------------------------------------------------------------
# If running from inside the repo, this can usually stay as None.
# If imports fail, set REPO_ROOT to your local clone, e.g.
# REPO_ROOT = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis")
REPO_ROOT = None

if REPO_ROOT is not None:
    src_path = Path(REPO_ROOT) / "src"
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

# Optional local copy of the summary table. The registry discovers sessions from BASE_PATH;
# this file is only useful for ad hoc inspection or manual cross-checks.
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")

# ---------------------------------------------------------------------
# Session selection
# ---------------------------------------------------------------------
target_mice = [
    826031,
    826032,
]

EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
PARADIGMS = ["change_detection_passive"]

In [4]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions for target mice: {target_mice}")
display(process_df.head())

Found 19 candidate sessions for target mice: [826031, 826032]


,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_id,camera_type,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
0,826031_2026-01-30_15-04-02,826031,2,2026-01-30,ASAP7y,NaN,50,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
1,826031_2026-02-01_11-01-50,826031,3,2026-02-01,ASAP7y,NaN,50,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
2,826031_2026-02-02_10-23-53,826031,4,2026-02-02,ASAP7y,NaN,50,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
3,826031_2026-02-03_14-21-45,826031,5,2026-02-03,ASAP7y,NaN,50,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
4,826031_2026-02-04_12-15-34,826031,6,2026-02-04,ASAP7y,NaN,50,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
